In [0]:
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, StringType

from src.config.settings import (
    LANDING_PATH,
    DATABASE_NAME,
    BRONZE_TABLE,
    SILVER_TABLE,
    QUARANTINE_TABLE,
    QUALITY_SUMMARY_TABLE,
    QUALITY_REJECTIONS_TABLE
)

from src.quality.quality_rules import (
    validate_required_columns,
    standardize_taxi_data,
    add_quality_validation_columns,
    get_valid_records,
    get_invalid_records
)

In [0]:
spark = SparkSession.builder.appName("ifood-case-taxi").getOrCreate()

In [0]:
def create_database() -> None:
    """
    Cria o database.
    """

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")

    print(f"Database {DATABASE_NAME} criado ou já existente.")


def drop_managed_tables_if_exists() -> None:
    """
    Remove tabelas gerenciadas antes da nova execução.
    Isso para evitar conflito de schema/location em reprocessamentos.
    Esta etapa está sendo aplicado para o estudo de caso apenas. 
    Se estivessemos em um ambiente produtivo poderiamos usar um DLT ou LDP
    """

    spark.sql(f"DROP TABLE IF EXISTS {QUALITY_REJECTIONS_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {QUALITY_SUMMARY_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {QUARANTINE_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")

    print("Tabelas antigas removidas")


def get_existing_column(df: DataFrame, column_name: str):
    """
    Retorna a coluna se existir no DataFrame.
    Caso contrário, retorna NULL.
    """

    if column_name in df.columns:
        return F.col(column_name)

    return F.lit(None)


def read_and_standardize_file(file_name: str) -> DataFrame:
    """
    Lê um arquivo Parquet individualmente e padroniza o schema.

    Essa abordagem evita conflitos de schema entre arquivos Yellow e Green Taxi,
    e também entre meses diferentes do mesmo tipo de táxi.
    """

    file_path = f"{LANDING_PATH}/{file_name}"

    print(f"Lendo arquivo local da Landing: {file_path}")

    df = (
        spark.read
        .format("parquet")
        .load(file_path)
    )

    validate_required_columns(df)

    taxi_type = "yellow" if file_name.startswith("yellow") else "green"

    pickup_datetime = (
        get_existing_column(df, "tpep_pickup_datetime")
        if taxi_type == "yellow"
        else get_existing_column(df, "lpep_pickup_datetime")
    )

    dropoff_datetime = (
        get_existing_column(df, "tpep_dropoff_datetime")
        if taxi_type == "yellow"
        else get_existing_column(df, "lpep_dropoff_datetime")
    )

    return (
        df
        .select(
            F.col("VendorID").cast(IntegerType()).alias("VendorID"),
            F.col("passenger_count").cast(DoubleType()).alias("passenger_count"),
            F.col("total_amount").cast(DoubleType()).alias("total_amount"),
            pickup_datetime.cast(TimestampType()).alias("pickup_datetime"),
            dropoff_datetime.cast(TimestampType()).alias("dropoff_datetime"),
            F.lit(taxi_type).cast(StringType()).alias("taxi_type"),
            F.lit(file_name).cast(StringType()).alias("source_file")
        )
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

def ingest_bronze() -> None:
    """
    Lê automaticamente os arquivos Parquet existentes na Landing,
    padroniza o schema e grava a camada Bronze como tabela gerenciada Delta.

    A função não depende mais de SOURCE_FILES.
    Ela identifica os arquivos já baixados na Landing.
    """

    print("Iniciando ingestão Bronze com padronização de schema.")

    try:
        landing_files_info = dbutils.fs.ls(LANDING_PATH)
    except Exception as error:
        raise FileNotFoundError(
            f"Não foi possível acessar a Landing: {LANDING_PATH}. "
            "Execute primeiro a etapa de ingestão para Landing."
        ) from error

    source_files = [
        file_info.name
        for file_info in landing_files_info
        if file_info.name.endswith(".parquet")
    ]

    source_files.sort()

    if not source_files:
        raise ValueError(
            f"Nenhum arquivo Parquet encontrado na Landing: {LANDING_PATH}"
        )

    print("Arquivos encontrados na Landing:")

    for file_name in source_files:
        print(f"- {file_name}")

    bronze_dataframes = []

    for file_name in source_files:
        print(f"Lendo arquivo da Landing: {file_name}")

        df_file = read_and_standardize_file(file_name)

        bronze_dataframes.append(df_file)

    df_bronze = reduce(
        lambda df1, df2: df1.unionByName(
            df2,
            allowMissingColumns=True
        ),
        bronze_dataframes
    )

    (
        df_bronze.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"Camada Bronze criada como tabela gerenciada: {BRONZE_TABLE}")

def build_silver_and_quarantine() -> None:
    """
    Lê a Bronze, aplica padronização, regras de qualidade
    e separa registros válidos e inválidos em tabelas.
    """

    df_bronze = spark.table(BRONZE_TABLE)

    df_standardized = standardize_taxi_data(df_bronze)

    df_validated = add_quality_validation_columns(df_standardized)

    df_valid = get_valid_records(df_validated)
    df_invalid = get_invalid_records(df_validated)

    (
        df_valid.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .partitionBy("pickup_month", "taxi_type")
        .saveAsTable(SILVER_TABLE)
    )

    (
        df_invalid.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .partitionBy("pickup_month", "taxi_type")
        .saveAsTable(QUARANTINE_TABLE)
    )

    print(f"Camada Silver criada: {SILVER_TABLE}")
    print(f"Camada Quarantine criada: {QUARANTINE_TABLE}")


def generate_quality_report() -> None:
    """
    Gera relatório de qualidade da carga.
    """

    df_valid = spark.table(SILVER_TABLE)
    df_invalid = spark.table(QUARANTINE_TABLE)

    valid_count = df_valid.count()
    invalid_count = df_invalid.count()
    total_count = valid_count + invalid_count

    valid_percentage = round((valid_count / total_count) * 100, 2) if total_count > 0 else 0
    invalid_percentage = round((invalid_count / total_count) * 100, 2) if total_count > 0 else 0

    summary_df = spark.createDataFrame(
        [
            (
                total_count,
                valid_count,
                invalid_count,
                valid_percentage,
                invalid_percentage
            )
        ],
        [
            "total_records",
            "valid_records",
            "invalid_records",
            "valid_percentage",
            "invalid_percentage"
        ]
    ).withColumn("report_timestamp", F.current_timestamp())

    rejection_df = (
        df_invalid
        .groupBy("pickup_month", "taxi_type", "rejection_reason")
        .agg(F.count("*").alias("rejected_records"))
        .orderBy("pickup_month", "taxi_type", F.desc("rejected_records"))
    )

    (
        summary_df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(QUALITY_SUMMARY_TABLE)
    )

    (
        rejection_df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(QUALITY_REJECTIONS_TABLE)
    )

    print(f"Tabela de resumo de qualidade criada: {QUALITY_SUMMARY_TABLE}")
    print(f"Tabela de rejeições criada: {QUALITY_REJECTIONS_TABLE}")


def landing_to_silver() -> None:
    print("Pre-loading - Criando database")
    create_database()

    print("Pre-loading - Limpando tabelas anteriores")
    drop_managed_tables_if_exists()

    print("Etapa 1 - Ingestão Bronze")
    ingest_bronze()

    print("Etapa 2 - Construção Silver e Quarantine")
    build_silver_and_quarantine()

    print("Etapa 3 - Relatório de qualidade")
    generate_quality_report()

    print("Pipeline executado com sucesso.")

In [0]:
landing_to_silver()